In [1]:
#BUILD 1
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 203kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.77MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 9.72MB/s]


### Custom Test Set Processing

In [19]:
import os

# 1. Find the file
print("Searching for test_set.zip...")
found_path = None
for root, dirs, files in os.walk('/content'):
    if 'test_set.zip' in files:
        found_path = os.path.join(root, 'test_set.zip')
        break

if not found_path:
    # Check root just in case
    if os.path.exists('/test_set.zip'):
        found_path = '/test_set.zip'

# 2. Extract it if found
if found_path:
    print(f"Success! Found file at: {found_path}")
    output_folder = 'test_set_images'
    os.makedirs(output_folder, exist_ok=True)
    !unzip -o "{found_path}" -d {output_folder}
    print("\nDone! You can now run the loading cell (786d274f).")
else:
    print("Could not find 'test_set.zip'. Please ensure the name is exactly 'test_set.zip' (case sensitive) and it finished uploading.")

Searching for test_set.zip...
Success! Found file at: /content/test_set.zip
Archive:  /content/test_set.zip
   creating: test_set_images/images/
  inflating: test_set_images/__MACOSX/._images  
  inflating: test_set_images/images/002.png  
  inflating: test_set_images/__MACOSX/images/._002.png  
  inflating: test_set_images/images/016.png  
  inflating: test_set_images/__MACOSX/images/._016.png  
  inflating: test_set_images/images/017.png  
  inflating: test_set_images/__MACOSX/images/._017.png  
  inflating: test_set_images/images/003.png  
  inflating: test_set_images/__MACOSX/images/._003.png  
  inflating: test_set_images/images/029.png  
  inflating: test_set_images/__MACOSX/images/._029.png  
  inflating: test_set_images/images/015.png  
  inflating: test_set_images/__MACOSX/images/._015.png  
  inflating: test_set_images/images/001.png  
  inflating: test_set_images/__MACOSX/images/._001.png  
  inflating: test_set_images/images/000.png  
  inflating: test_set_images/__MACOSX/i

In [23]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import os

# The unzip created 'test_set_images/images/'
# ImageFolder needs the path to the folder CONTAINING the category folders
base_path = 'test_set_images'

if not os.path.exists(os.path.join(base_path, 'images')):
    print(f"Error: Could not find the 'images' folder inside '{base_path}'.")
else:
    test_transforms = v2.Compose([
        v2.Resize((28, 28)),
        v2.Grayscale(num_output_channels=1),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ])

    # root=base_path will look inside for the 'images' folder and treat it as a class
    custom_test_dataset = datasets.ImageFolder(
        root=base_path,
        transform=test_transforms
    )

    custom_test_dataloader = DataLoader(custom_test_dataset, batch_size=64, shuffle=False)

    print(f"Successfully loaded {len(custom_test_dataset)} images from {base_path}/images/")

Successfully loaded 200 images from test_set_images/images/


In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.relu_stack = nn.Sequential(
        nn.Linear(28*28, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,32),
        nn.ReLU(),
        nn.Linear(32,10)
    )
  def forward(self, x):
    x = self.flatten(x)
    logits = self.relu_stack(x)
    return logits

ffn_model = NeuralNetwork().to(device)

In [5]:
loss_fn = nn.CrossEntropyLoss()


def train(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset)
  model.train()
  for batch, (X,y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)
    pred = model(X)
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
def test(dataloader, model,loss_fn):
  model.eval()
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  test_loss, correct = 0, 0
  with torch.no_grad():
    for X,y in dataloader:
      X,y = X.to(device), y.to(device)
      pred = model(X)
      test_loss+=loss_fn(pred, y).item()
      correct += (pred.argmax(1) == y).type(torch.float).sum().item()
  test_loss /= num_batches
  correct /= size
  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [6]:
ffn_optimizer = torch.optim.AdamW(ffn_model.parameters(), lr=1e-4)
epochs = 12
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, ffn_model, loss_fn, ffn_optimizer)
    test(test_dataloader, ffn_model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.297005  [    0/60000]
loss: 1.621856  [ 6400/60000]
loss: 0.873169  [12800/60000]
loss: 0.920570  [19200/60000]
loss: 0.830298  [25600/60000]
loss: 0.646160  [32000/60000]
loss: 0.776598  [38400/60000]
loss: 0.748672  [44800/60000]
loss: 0.678054  [51200/60000]
loss: 0.624107  [57600/60000]
Test Error: 
 Accuracy: 76.6%, Avg loss: 0.652478 

Epoch 2
-------------------------------
loss: 0.576052  [    0/60000]
loss: 0.712967  [ 6400/60000]
loss: 0.454294  [12800/60000]
loss: 0.613074  [19200/60000]
loss: 0.585358  [25600/60000]
loss: 0.452784  [32000/60000]
loss: 0.545067  [38400/60000]
loss: 0.712109  [44800/60000]
loss: 0.571638  [51200/60000]
loss: 0.506026  [57600/60000]
Test Error: 
 Accuracy: 80.8%, Avg loss: 0.534492 

Epoch 3
-------------------------------
loss: 0.426057  [    0/60000]
loss: 0.561771  [ 6400/60000]
loss: 0.388538  [12800/60000]
loss: 0.555132  [19200/60000]
loss: 0.499886  [25600/60000]
loss: 0.409664  [32000/600

In [7]:
#Build - 2
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

In [8]:
from torch.nn.modules.activation import ReLU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv_stack = nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

    self.flatten = nn.Flatten()
    self.classifier = nn.Sequential(
        nn.Linear(64 * 7 * 7, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 10)
    )

  def forward(self, x):
    x = self.conv_stack(x)
    x = self.flatten(x)
    logits = self.classifier(x)
    return logits

cnn_model = NeuralNetwork().to(device)

In [9]:
epochs = 12

cnn_optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=1e-4)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, cnn_model, loss_fn, cnn_optimizer)
    test(test_dataloader, cnn_model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.299363  [    0/60000]
loss: 1.491589  [ 6400/60000]
loss: 0.940683  [12800/60000]
loss: 1.027929  [19200/60000]
loss: 0.789144  [25600/60000]
loss: 0.875829  [32000/60000]
loss: 0.742582  [38400/60000]
loss: 0.770038  [44800/60000]
loss: 0.632279  [51200/60000]
loss: 0.699264  [57600/60000]
Test Error: 
 Accuracy: 76.6%, Avg loss: 0.643663 

Epoch 2
-------------------------------
loss: 0.562933  [    0/60000]
loss: 0.786879  [ 6400/60000]
loss: 0.444520  [12800/60000]
loss: 0.660205  [19200/60000]
loss: 0.619714  [25600/60000]
loss: 0.531402  [32000/60000]
loss: 0.547917  [38400/60000]
loss: 0.628310  [44800/60000]
loss: 0.567710  [51200/60000]
loss: 0.567192  [57600/60000]
Test Error: 
 Accuracy: 80.5%, Avg loss: 0.539407 

Epoch 3
-------------------------------
loss: 0.459189  [    0/60000]
loss: 0.649953  [ 6400/60000]
loss: 0.354831  [12800/60000]
loss: 0.571829  [19200/60000]
loss: 0.536264  [25600/60000]
loss: 0.427027  [32000/600

### Processing and Predicting on Custom Test Set

In [21]:
import os

# Create a directory to extract the images into
output_folder = 'test_set_images'
os.makedirs(output_folder, exist_ok=True)

# Unzip the file into the new directory
print(f"Unzipping test_set.zip into {output_folder}/")
!unzip -o test_set.zip -d {output_folder}

Unzipping test_set.zip into test_set_images/
Archive:  test_set.zip
  inflating: test_set_images/__MACOSX/._images  
  inflating: test_set_images/images/002.png  
  inflating: test_set_images/__MACOSX/images/._002.png  
  inflating: test_set_images/images/016.png  
  inflating: test_set_images/__MACOSX/images/._016.png  
  inflating: test_set_images/images/017.png  
  inflating: test_set_images/__MACOSX/images/._017.png  
  inflating: test_set_images/images/003.png  
  inflating: test_set_images/__MACOSX/images/._003.png  
  inflating: test_set_images/images/029.png  
  inflating: test_set_images/__MACOSX/images/._029.png  
  inflating: test_set_images/images/015.png  
  inflating: test_set_images/__MACOSX/images/._015.png  
  inflating: test_set_images/images/001.png  
  inflating: test_set_images/__MACOSX/images/._001.png  
  inflating: test_set_images/images/000.png  
  inflating: test_set_images/__MACOSX/images/._000.png  
  inflating: test_set_images/images/014.png  
  inflating: 

#### Generate Predictions for Custom Test Set using FFN Model

In [25]:
import pandas as pd
import torch
import os

# Check if the dataloader exists before proceeding
if 'custom_test_dataloader' not in locals():
    print("Error: 'custom_test_dataloader' is not defined. Please ensure the data loading cell (786d274f) has run successfully.")
else:
    ffn_model.to(device)
    ffn_model.eval()

    custom_ffn_predictions = []
    custom_ffn_image_filenames = []

    # ImageFolder might include the __MACOSX files which cause crashes.
    # We will iterate through the dataset and only predict on valid images.
    with torch.no_grad():
        for i in range(len(custom_test_dataset)):
            image_path, _ = custom_test_dataset.samples[i]
            filename = os.path.basename(image_path)

            # Skip hidden macOS metadata files
            if filename.startswith('._'):
                continue

            # Load and predict individual image
            input_tensor, _ = custom_test_dataset[i]
            input_tensor = input_tensor.unsqueeze(0).to(device) # Add batch dimension

            output = ffn_model(input_tensor)
            pred = output.argmax(1).item()

            custom_ffn_predictions.append(pred)
            custom_ffn_image_filenames.append(os.path.splitext(filename)[0])

    # Create a DataFrame
    custom_ffn_results_df = pd.DataFrame({
        'image_id': custom_ffn_image_filenames,
        'label': custom_ffn_predictions
    })

    custom_ffn_output_filename = '2025CS11175_NN.csv'
    custom_ffn_results_df.to_csv(custom_ffn_output_filename, index=False)

    print(f"Successfully processed {len(custom_ffn_predictions)} valid images.")
    print(f"Predictions saved to {custom_ffn_output_filename}")
    display(custom_ffn_results_df.head())

Successfully processed 100 valid images.
Predictions saved to 2025CS11175_NN.csv


,image_id,label
0,000,5
1,001,3
2,002,7
3,003,4
4,004,3


#### Generate Predictions for Custom Test Set using CNN Model

In [27]:
import pandas as pd
import torch
import os

# Safety check: only run if the dataloader was successfully created
if 'custom_test_dataloader' not in locals():
    print("Error: 'custom_test_dataloader' is not defined. Please ensure the data loading cell (786d274f) runs successfully.")
else:
    # Ensure the cnn_model is in evaluation mode and on the correct device
    cnn_model.to(device)
    cnn_model.eval()

    custom_cnn_predictions = []
    custom_cnn_image_filenames = []

    # Iterate through the dataset manually to skip hidden macOS files
    with torch.no_grad():
        for i in range(len(custom_test_dataset)):
            image_path, _ = custom_test_dataset.samples[i]
            filename = os.path.basename(image_path)

            # Skip hidden macOS metadata files
            if filename.startswith('._'):
                continue

            # Load and predict individual image
            input_tensor, _ = custom_test_dataset[i]
            input_tensor = input_tensor.unsqueeze(0).to(device)

            output = cnn_model(input_tensor)
            pred = output.argmax(1).item()

            custom_cnn_predictions.append(pred)
            custom_cnn_image_filenames.append(os.path.splitext(filename)[0])

    # Create a DataFrame with image IDs and predictions for CNN
    custom_cnn_results_df = pd.DataFrame({
        'image_id': custom_cnn_image_filenames,
        'label': custom_cnn_predictions
    })

    custom_cnn_output_filename = '2025CS11175_CNN.csv'
    custom_cnn_results_df.to_csv(custom_cnn_output_filename, index=False)

    print(f"Successfully processed {len(custom_cnn_predictions)} valid images for CNN.")
    print(f"Predictions saved to {custom_cnn_output_filename}")
    display(custom_cnn_results_df.head())

Successfully processed 100 valid images for CNN.
Predictions saved to 2025CS11175_CNN.csv


,image_id,label
0,000,5
1,001,1
2,002,7
3,003,4
4,004,3
